# LLaMA

**LLaMA** is Meta's family of **open-weight** large language models (Llama 1 → 2 → 3 / 3.1 / 3.2 / 3.3 → Llama 4). Unlike the closed APIs in this domain, you **download the weights** from Hugging Face and run them yourself — locally, on-prem, or on your own cloud — via `transformers`, `llama.cpp`/Ollama, vLLM, or MLX.

**Domain:** Proprietary Models & Coding AI  ·  **from study list**  ·  **runnable:** yes  ·  _HF weights (gated; live cells gate on `os.getenv`)_

## 1. What & Why

**LLaMA** (Large Language Model Meta AI) is the open-weight LLM line from **Meta**. The defining trait versus everything else in this domain (Grok, Claude, Gemini, …): **you get the weights**. You download them and run inference on your own hardware, fine-tune them, quantize them, and ship them offline.

**The roster you actually meet today:**

- **Llama 3.1** — `8B`, `70B`, `405B`; 128K context; the workhorse generation.
- **Llama 3.2** — small `1B` / `3B` text models (great on laptops/edge) plus `11B` / `90B` **vision** models.
- **Llama 3.3 70B** — a 70B tuned to land near 405B quality at a fraction of the cost.
- **Llama 4** (Scout / Maverick) — newer **mixture-of-experts**, multimodal, very long context.

Each size ships as a **base** model and an **Instruct** (chat-tuned) model.

**The problem it solves / why reach for it:**

- **Self-hosting & data residency.** The prompt and the weights never leave your network. For regulated data or air-gapped deployments, that's the whole ballgame.
- **Cost at scale.** Once you own the GPUs (or rent them), there's no per-token bill. High-volume workloads can be dramatically cheaper than a metered API.
- **Customization.** Open weights mean **fine-tuning** (LoRA/QLoRA), quantization, distillation, and full control over the serving stack — none of which a closed API gives you.
- **No vendor lock-in / offline.** It runs without a network. Your app doesn't die when a provider changes pricing, deprecates a model, or rate-limits you.

**When NOT to:** if you want the **absolute frontier with zero ops**, a hosted Claude/GPT call is simpler and often smarter than a 70B you babysit. If you have no GPU and low volume, a metered API is cheaper than idle hardware. And note the license is **"open weight," not OSI open source** — the Llama Community License adds use restrictions (notably a >700M-monthly-active-user clause and acceptable-use terms), so read it before you build a business on it.

## 2. Mental Model

Think of LLaMA as **"weights in a file you load into a runtime you choose."** There is no `api.llama.com` you call — *you* are the inference provider.

```
   Meta releases weights ──> Hugging Face Hub (gated: accept license + HF token)
                                          │
                       download (safetensors / GGUF)
                                          │
        ┌─────────────────┬───────────────┴───────────┬──────────────────┐
        ▼                 ▼                            ▼                  ▼
   transformers      llama.cpp / Ollama            vLLM / TGI           MLX
   (research, FT)    (laptop, GGUF, CPU/GPU)     (high-throughput     (Apple
        │                 │                         serving)           Silicon)
        └─────────────────┴───────────────┬───────────┴──────────────────┘
                                          ▼
                       your hardware (GPU VRAM / CPU RAM)
                                          │
                                          ▼
                       tokens out — you own the whole pipeline
```

Three things to internalize:

1. **You pick the runtime, not a base_url.** The same weights run under `transformers` (flexible, fine-tuning), `llama.cpp`/**Ollama** (easy local, GGUF quantization), **vLLM**/TGI (production throughput), or **MLX** (Mac). The choice is an ops decision, not a model decision.
2. **The chat template is part of the model.** Instruct models expect a precise stream of **special tokens** (`<|begin_of_text|>`, `<|start_header_id|>…<|end_header_id|>`, `<|eot_id|>`). Get it wrong and quality silently tanks — so you call `tokenizer.apply_chat_template(...)`, you don't hand-concatenate strings.
3. **Memory is the gating constraint.** Whether a model fits is roughly `params × bytes-per-weight`. Quantization (FP16 → 8-bit → 4-bit) is the lever that turns "needs an A100" into "runs on my laptop," at some quality cost.

## 3. Key Concepts

- **Open weights** — Meta publishes the actual model parameters. You download and run them; contrast with closed APIs where you only ever see outputs.
- **Llama Community License** — permissive but **not** OSI open source: acceptable-use policy + a clause requiring a separate license if your product has **>700M monthly active users**. Fine for the overwhelming majority of uses; read it anyway.
- **Gated access** — on Hugging Face you must **accept the license** for the repo (e.g. `meta-llama/Llama-3.1-8B-Instruct`) and pass an **HF token** (`huggingface-cli login`) to download. Community re-uploads (e.g. `NousResearch/...`) are sometimes ungated mirrors.
- **Base vs Instruct** — *base* is a raw next-token predictor (good for fine-tuning / completion); ***Instruct*** is chat/RLHF-tuned and the one you want for assistants. Don't chat with a base model.
- **Chat template & special tokens** — Llama 3's format uses `<|begin_of_text|>`, `<|start_header_id|>{role}<|end_header_id|>`, message text, then `<|eot_id|>`. Always generate it with `tokenizer.apply_chat_template`.
- **Tokenizer** — Llama 1/2 used a 32K **SentencePiece** vocab; **Llama 3 switched to a 128K tiktoken-style BPE**, which improves efficiency (fewer tokens per word) and multilingual coverage. Tokenizers are *not* interchangeable across generations.
- **Context length** — 4K (Llama 2) → **128K** (Llama 3.1+). Longer context costs more KV-cache memory.
- **Quantization** — store weights in fewer bits to shrink memory: **FP16** (2 B/param) → **8-bit** (1 B) → **4-bit** (~0.5 B). Formats: **GGUF** (llama.cpp/Ollama), **bitsandbytes** (transformers), **AWQ/GPTQ** (calibrated 4-bit), **MLX** (Mac).
- **Runtimes** — `transformers` (HF, research + fine-tuning), **llama.cpp/Ollama** (local, GGUF, CPU-friendly), **vLLM**/TGI (high-throughput serving with paged-attention), **MLX** (Apple Silicon).
- **Fine-tuning** — **LoRA / QLoRA** train a small set of adapter weights on top of a (often quantized) frozen base, making single-GPU fine-tuning of large models feasible.
- **VRAM rule of thumb** — inference needs roughly `params × bytes/param` for weights, **plus** KV-cache that grows with context × batch. An 8B at 4-bit fits comfortably in 8 GB; a 70B at 4-bit needs ~40 GB+.

## 4. Setup

Two common paths. **Ollama** is the fastest way to *run* Llama; **transformers** is the path for fine-tuning and research.

```bash
# Path A — Ollama (easiest local inference; bundles a quantized GGUF)
#   install from https://ollama.com, then:
ollama run llama3.2:3b        # downloads + chats, CPU-OK

# Path B — Hugging Face transformers (research / fine-tuning)
pip install "transformers>=4.43" torch accelerate
pip install bitsandbytes        # optional: 4-bit / 8-bit quantization (CUDA)

# Gated Llama repos require accepting the license + an HF token:
huggingface-cli login           # paste a token from https://huggingface.co/settings/tokens
```

A minimal `transformers` generation looks like this (gated models download several GB, so the live cell below is guarded):

```python
from transformers import pipeline
pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")
out = pipe([{"role": "user", "content": "Hello"}], max_new_tokens=32)
print(out[0]["generated_text"][-1]["content"])
```

The cells below run top-to-bottom in a fresh kernel **without** `torch`, the weights, or an HF token — every download/inference call is gated behind an `os.getenv("RUN_LLAMA")` (or token) check, while the no-network examples always execute.

In [ ]:
# This notebook executes with or without transformers, weights, or an HF token.
# To run the live example: set RUN_LLAMA=1 and (for gated repos) HF_TOKEN, then
#   pip install "transformers>=4.43" torch accelerate
import os

hf_token   = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
run_llama  = os.getenv("RUN_LLAMA") == "1"
# A *small, ungated* model keeps the live demo CPU-friendly and license-free.
demo_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # Llama-architecture, not gated

try:
    import torch  # noqa: F401
    have_torch = True
except ImportError:
    have_torch = False

print("RUN_LLAMA   :", "set" if run_llama else "(unset — live inference skipped)")
print("HF_TOKEN    :", "set" if hf_token else "(unset — gated repos unavailable)")
print("torch       :", "installed" if have_torch else "(not installed)")
print("demo model  :", demo_model)

## 5. Worked Examples

### Example 1 — Build the Llama 3 chat prompt by hand (no network)

The single most important thing to know about running Llama is its **chat format**. Instruct models were trained on a specific stream of special tokens; feed them anything else and quality quietly collapses. Below we construct the exact string `tokenizer.apply_chat_template` would produce, so the format is in muscle memory.

In [ ]:
# The Llama 3 instruct chat format — pure string assembly, no model needed.
BOS, EOT = "<|begin_of_text|>", "<|eot_id|>"

def llama3_prompt(messages):
    """Reproduce tokenizer.apply_chat_template(..., add_generation_prompt=True)."""
    out = [BOS]
    for m in messages:
        out.append(f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n"
                   f"{m['content']}{EOT}")
    # add_generation_prompt: open an empty assistant turn for the model to fill
    out.append("<|start_header_id|>assistant<|end_header_id|>\n\n")
    return "".join(out)

messages = [
    {"role": "system", "content": "You are a terse assistant."},
    {"role": "user",   "content": "Name the capital of France."},
]
prompt = llama3_prompt(messages)
print(prompt)
print("\n--- the model now generates here, and STOPS when it emits <|eot_id|> ---")

### Example 2 — Will it fit? Memory estimate by size & quantization (no network)

Whether a model runs on your box is mostly arithmetic: `params × bytes-per-weight` for the weights, plus KV-cache for context. This estimator is the back-of-envelope you'll reach for constantly when picking a size/quant combo for given VRAM.

In [ ]:
# Rough inference-memory estimator for Llama models. Pure Python, no GPU.
PARAMS_B = {"1B": 1.24, "3B": 3.21, "8B": 8.03, "70B": 70.6, "405B": 405.0}  # billions
BYTES_PER_WEIGHT = {"fp16": 2.0, "int8": 1.0, "int4": 0.5}  # quantization → B/param

def weight_gb(size, quant):
    return PARAMS_B[size] * 1e9 * BYTES_PER_WEIGHT[quant] / 1024**3

def fits(gb, vram_gb):
    # ~20% overhead for activations + KV-cache + CUDA context
    return gb * 1.20 <= vram_gb

print(f"{'model':>6} | {'fp16':>8} {'int8':>8} {'int4':>8}   (GB of weights)")
print("-" * 44)
for size in PARAMS_B:
    row = "  ".join(f"{weight_gb(size, q):6.1f}" for q in BYTES_PER_WEIGHT)
    print(f"{size:>6} |   {row}")

print("\nFit check on a 24 GB GPU (e.g. RTX 4090):")
for size in ("8B", "70B"):
    for quant in ("fp16", "int4"):
        gb = weight_gb(size, quant)
        print(f"  Llama-{size:<4} {quant:>4}: {gb:6.1f} GB -> "
              f"{'fits' if fits(gb, 24) else 'too big'}")

### Example 3 — Load a tiny Llama-architecture model and generate (gated)

The real `transformers` path. We use **TinyLlama-1.1B** — same Llama architecture, but small, CPU-runnable, and **ungated** so there's no license/token friction. With `RUN_LLAMA=1` and `torch` installed it downloads (~2 GB) and generates real text; otherwise it prints the call shape so the notebook still executes cleanly. For the *actual* Meta weights, swap in `meta-llama/Llama-3.2-1B-Instruct` (requires accepting the license + `HF_TOKEN`).

In [ ]:
import os

def generate(prompt, model_id=demo_model, max_new_tokens=40):
    from transformers import pipeline          # pip install transformers torch
    pipe = pipeline("text-generation", model=model_id)   # downloads on first call
    out = pipe(
        [{"role": "user", "content": prompt}],  # pipeline applies the chat template
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return out[0]["generated_text"][-1]["content"]

if run_llama and have_torch:
    try:
        print("model says:", generate("In one sentence, what is a llama?").strip())
    except Exception as e:                       # offline / OOM / download failure
        print("Live run failed:", type(e).__name__, e)
else:
    print("Skipping live inference (set RUN_LLAMA=1 and install torch to run).")
    print("Call shape:")
    print("  pipe = pipeline('text-generation', model='meta-llama/Llama-3.2-1B-Instruct')")
    print("  pipe([{'role':'user','content': ...}], max_new_tokens=40)")
    print("Note: gated Meta repos also need huggingface-cli login (HF_TOKEN).")

### Example 4 — The Ollama path: easiest local serving (call shape, gated)

For "just run Llama on my machine," **Ollama** is usually the answer: one binary, automatic GGUF quantization, an OpenAI-compatible HTTP endpoint. The shape below is what you'll type and call; it's gated since it needs the Ollama daemon running with a model pulled.

In [ ]:
import os, json, urllib.request

OLLAMA_URL = "http://localhost:11434/api/chat"

def ollama_chat(prompt, model="llama3.2:3b"):
    body = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }).encode()
    req = urllib.request.Request(OLLAMA_URL, data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)["message"]["content"]

if os.getenv("RUN_OLLAMA") == "1":
    try:
        print("ollama says:", ollama_chat("Reply with one word: pong").strip())
    except Exception as e:
        print("Ollama call failed:", type(e).__name__, e)
else:
    print("Skipping Ollama call (set RUN_OLLAMA=1 with the daemon running).")
    print("Setup:  ollama run llama3.2:3b        # pull + chat from the CLI")
    print("HTTP :  POST http://localhost:11434/api/chat  {model, messages, stream}")
    print("Also :  Ollama exposes an OpenAI-compatible /v1/chat/completions endpoint.")

## 6. Gotchas & Pitfalls

- **Wrong chat template = silent quality collapse.** The #1 mistake. Instruct models need the exact special-token stream; hand-rolled `"User: ... Assistant:"` strings or the wrong generation's template produce rambling, looping, or refusing output. **Always** use `tokenizer.apply_chat_template`, and make sure the tokenizer matches the model.
- **Chatting with a *base* model.** Base checkpoints aren't instruction-tuned — they just continue text. If your "assistant" ignores instructions and free-associates, check you didn't load the non-Instruct repo.
- **Gated-repo 401s.** `meta-llama/*` downloads fail until you **accept the license on the model page** *and* `huggingface-cli login`. Approval can take time; until then use an ungated mirror or a different size.
- **Underestimating VRAM (and KV-cache).** Weight size is only part of it — long context and batching grow the KV-cache fast. A model that "fits" at 2K context can OOM at 128K. Leave headroom or quantize.
- **Quantization isn't free.** 4-bit roughly halves quality-sensitive tasks' margin vs FP16. For most chat it's fine; for hard reasoning/coding, test int4 vs int8/FP16 before committing. GPTQ/AWQ (calibrated) usually beat naive int4.
- **Forgetting the EOS / stop token.** Llama 3 ends turns with `<|eot_id|>`, not the classic `</s>`. If generation runs on forever, your stop-token config is wrong for the generation you loaded.
- **Tokenizer mismatch across versions.** Llama 2 (SentencePiece, 32K) and Llama 3 (BPE, 128K) tokenizers are **not** interchangeable. Mixing a Llama 2 tokenizer with Llama 3 weights yields garbage.
- **"Open" ≠ unrestricted.** The Community License has an acceptable-use policy and the >700M-MAU clause. It's not Apache/MIT — don't assume you can do literally anything.
- **`device_map="auto"` surprises.** Without it (and `accelerate`), big models try to load onto one device and OOM; with it, layers can spill to CPU/disk and inference crawls. Know where your weights actually live.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Self-hosting / data never leaves your network** | **Llama** | Open weights you run on your own hardware; closed APIs can't offer this. |
| **Fine-tuning on your own data** | **Llama** (LoRA/QLoRA) | Full access to weights; train adapters on a single GPU. |
| **Cheapest high-volume inference** | **Llama** (self-hosted) | No per-token bill once you own/rent the GPUs. |
| **Absolute frontier quality, zero ops** | **Claude / GPT** (closed APIs) | Hosted frontier models beat a 70B you self-manage, with no infra. |
| **Tiny local / edge / laptop model** | **Llama 3.2 1B/3B**, **Qwen**, **Mistral** | Small open models run on CPU / phones; pick on benchmarks for your task. |
| **Open weights, different trade-offs** | **Mistral / Qwen / DeepSeek / Gemma** | Other open families — Qwen is strong multilingual/coding, Mistral MoE is efficient, DeepSeek leads on reasoning-per-dollar. |

**Honest trade-offs:**

- **vs closed APIs (Claude, GPT, Grok, Gemini)** — Llama trades some peak quality and *all* the ops convenience for **control, privacy, cost-at-scale, and offline use**. If you don't need those four, a hosted API is usually the better default.
- **vs Mistral / Qwen / DeepSeek / Gemma** (other open weights) — these compete directly with Llama; the right pick is task- and license-specific. Qwen often leads coding/multilingual, Mistral's MoE is compute-efficient, DeepSeek is strong on reasoning value, Gemma is Google's small-model line. **Benchmark on *your* task** rather than trusting the leaderboard.
- **vs running Llama through a hosted provider** (Together, Fireworks, Groq, AWS Bedrock) — you can rent Llama inference instead of self-hosting: open-model flexibility (and model choice) without owning GPUs, at a per-token price. A sensible middle ground when you want Llama but not the ops.

**Rule of thumb:** reach for Llama when **control, privacy, customization, or volume economics** matter — and you're willing to own (or rent) the serving stack. Default to a closed frontier API when you just want the smartest possible answer with no infrastructure.

## 8. Resources

- **Llama official site & model cards** — https://www.llama.com/
- **Meta Llama on Hugging Face (gated repos)** — https://huggingface.co/meta-llama
- **Llama 3 model card & prompt format** — https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_1/
- **`transformers` text-generation docs** — https://huggingface.co/docs/transformers/en/llm_tutorial
- **Ollama (easiest local inference)** — https://ollama.com/
- **llama.cpp (GGUF / CPU inference)** — https://github.com/ggml-org/llama.cpp
- **vLLM (high-throughput serving)** — https://docs.vllm.ai/
- **Llama Community License & Acceptable Use Policy** — https://www.llama.com/llama3_1/license/

**Related notebooks:** `mistral`, `qwen`, `deepseek` (other open-weight families for head-to-head); the closed APIs `anthropic-claude-api`, `google-gemini`, `grok` for the self-host-vs-hosted trade-off.

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def plan_fit(params_b, vram_gb, context_tokens=0, kv_gb_per_1k=0.0, overhead=0.20):
    """The highest-precision quantization that fits, or None with the smallest total."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE